# 01 - Merge and prep
This file merges all csv files per crisis and adds a label of what phase the comment was made

Merges the five phase result files into one table (`analysis_df.csv`) for `02-statistical-analysis`.
Run from the repo root. Check the paths and `OK_STATUS` in the config cell first.

In [1]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd

os.chdir(r"C:\Users\PHSP8131\crisis-management-analysis")

ARCHIVE = Path("03-llm-passes/archive/2026-10-05_v1.4_4bit")
CRISIS = "ateneo-drowning"
PHASES = ["01-onset", "02-confirmation", "03-spread", "04-reckoning", "05-lingering"]
OK_STATUS = None
OUT = Path("04-statistical-analysis/output") / CRISIS
OUT.mkdir(parents=True, exist_ok=True)

## Load

In [7]:
from pathlib import Path
import os
os.chdir(r"C:\Users\PHSP8131\crisis-management-analysis")
base = ARCHIVE / CRISIS / "full-codebook-coding"
frames = []
for p in PHASES:
    f = base / p / "full_codebook_results.csv"
    print(p, f.exists())
    d = pd.read_csv(f, keep_default_na=False, na_values=[""])
    frames.append(d.assign(phase=p, source_file=f.name))
df = pd.concat(frames, ignore_index=True)
df.shape

01-onset True
02-confirmation True
03-spread True
04-reckoning True
05-lingering True


(11561, 47)

## Checks before filtering

In [8]:
print(df.groupby("phase").size(), "\n")
for c in ["status", "autofix", "confidence", "LANGUAGE"]:
    print(df[c].value_counts(dropna=False), "\n")
print("duplicate comment_id:", df.duplicated("comment_id").sum())

phase
01-onset            856
02-confirmation    2652
03-spread          5380
04-reckoning       1411
05-lingering       1262
dtype: int64 

status
coded      10140
missing     1421
Name: count, dtype: int64 

autofix
NaN                                                                                   9173
language_check:no_filipino_words                                                      1203
stance->NEUTRAL(needs_context)                                                         642
reject_reasons->NA                                                                     299
language_check:filipino_words_in_english                                               129
language_check:no_filipino_words;stance->NEUTRAL(needs_context)                         37
stance->NEUTRAL(needs_context);reject_reasons->NA                                       30
language_check:no_filipino_words;reject_reasons->NA                                     29
language_check:filipino_words_in_english;stance->NEUTR

## Filter

In [9]:
if OK_STATUS is not None:
    df = df[df.status == OK_STATUS]
df = df[pd.to_numeric(df.UNCODABLE, errors="coerce").fillna(0) == 0].copy()
len(df)

11551

## Types and derived variables

In [10]:
binary = ["ANGER", "CONTEMPT", "DISGUST", "SADNESS", "CONDOLENCE", "COMPASSION_VICTIMS",
          "SYMPATHY_OFFENDER", "FEAR", "NORM_VIOLATION", "IGNORED_WARNING", "RECURRENCE",
          "COVER_UP", "ALT_ACCOUNT", "INJUSTICE", "SELF_INTEREST", "NEEDS_CONTEXT",
          "CLASS_FRAME", "APPEARANCE_ATTACK", "META_DISCOURSE", "SARCASM"]
cats = ["ATTRIBUTION", "STANCE", "TARGET_OF_COMMENT", "CLASS_DIRECTION", "RESPONSE_TARGET",
        "RESPONSE_EVAL", "LANGUAGE", "confidence"]

df[binary] = df[binary].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
for c in cats:
    df[c] = df[c].fillna("NA").astype(str).str.strip()

df["phase"] = pd.Categorical(df.phase, PHASES, ordered=True)
df["phase_num"] = df.phase.cat.codes

emotions = ["ANGER", "CONTEMPT", "DISGUST", "SADNESS", "CONDOLENCE",
            "COMPASSION_VICTIMS", "SYMPATHY_OFFENDER", "FEAR"]
df["NEG_EMOTION"] = df[["ANGER", "CONTEMPT", "DISGUST"]].max(axis=1)
df["ANY_EMOTION"] = df[emotions].max(axis=1)
df["log_len"] = np.log1p(df.text.str.len())

## Split multi-label columns

In [11]:
multi = ["BLAME_TARGETS", "VALUES", "ACTIONS", "IDENTITY_FRAME", "REJECT_REASONS"]

def parse(x):
    if pd.isna(x):
        return []
    s = re.sub(r"[\[\]'\"]", "", str(x))
    return [t.strip() for t in re.split(r"[|;,]", s) if t.strip()]

for c in multi:
    tokens = df[c].apply(parse)
    for v in sorted({t for ts in tokens for t in ts} - {"NA"}):
        df[f"{c}__{v}"] = tokens.apply(lambda ts: int(v in ts))

[c for c in df.columns if "__" in c]

['BLAME_TARGETS__AUTHORITIES',
 'BLAME_TARGETS__COACH',
 'BLAME_TARGETS__COACHING_STAFF',
 'BLAME_TARGETS__NONE',
 'BLAME_TARGETS__ORGANIZERS',
 'BLAME_TARGETS__OTHER',
 'BLAME_TARGETS__PRESIDENT',
 'BLAME_TARGETS__UNIVERSITY',
 'BLAME_TARGETS__VENUE',
 'BLAME_TARGETS__VICTIMS',
 'VALUES__CHILD_POTENTIAL',
 'VALUES__DUTY_OF_CARE',
 'VALUES__EDUCATION_MOBILITY',
 'VALUES__FAITH',
 'VALUES__FAMILY',
 'VALUES__JUSTICE',
 'VALUES__NONE',
 'VALUES__OTHER',
 'VALUES__PARENT_SACRIFICE',
 'VALUES__SANCTITY_OF_LIFE',
 'VALUES__SOLIDARITY',
 'ACTIONS__ACCOUNTABILITY',
 'ACTIONS__BOYCOTT',
 'ACTIONS__INFORMATION',
 'ACTIONS__NONE',
 'ACTIONS__PUNITIVE',
 'ACTIONS__REFORM',
 'ACTIONS__RESTITUTION',
 'ACTIONS__RETALIATION',
 'ACTIONS__SUPPORT',
 'IDENTITY_FRAME__NATIONALITY',
 'IDENTITY_FRAME__NONE',
 'IDENTITY_FRAME__RACE',
 'IDENTITY_FRAME__REGION',
 'IDENTITY_FRAME__RELIGION',
 'REJECT_REASONS__INSINCERE',
 'REJECT_REASONS__NO_FAMILY_OUTREACH',
 'REJECT_REASONS__NO_INFORMATION',
 'REJECT_REASONS

## Broad outrage

In [12]:
blame = [c for c in df if c.startswith("BLAME_TARGETS__") and not c.endswith("__NONE")]
acts = [c for c in ["ACTIONS__PUNITIVE", "ACTIONS__RETALIATION"] if c in df]
targeted = (df[blame].max(axis=1).eq(1) | df.ATTRIBUTION.eq("ACCUSATION")
            | df[acts].max(axis=1).eq(1))
df["BROAD_OUTRAGE"] = (df.NEG_EMOTION.eq(1) & targeted).astype(int)

In [13]:
no_dir = df.CLASS_DIRECTION.eq("NA")
resp_t, resp_e = df.RESPONSE_TARGET.eq("NA"), df.RESPONSE_EVAL.eq("NA")
rules = {
    "direction without class frame": df.CLASS_FRAME.eq(0) & ~no_dir,
    "class frame without direction": df.CLASS_FRAME.eq(1) & no_dir,
    "response target/eval mismatch": resp_t != resp_e,
    "needs_context but stance not neutral": df.NEEDS_CONTEXT.eq(1) & df.STANCE.ne("NEUTRAL"),
}
for k, m in rules.items():
    print(k, int(m.sum()))
df["flag_inconsistent"] = pd.concat(rules.values(), axis=1).any(axis=1).astype(int)

direction without class frame 0
class frame without direction 0
response target/eval mismatch 0
needs_context but stance not neutral 0


In [14]:
print(pd.crosstab(df.phase, df.CLASS_FRAME, margins=True), "\n")
print(pd.crosstab(df.phase, df.CLASS_DIRECTION), "\n")
print(pd.crosstab(df.CLASS_FRAME, df.NEG_EMOTION, margins=True))

CLASS_FRAME          0    1    All
phase                             
01-onset           825   30    855
02-confirmation   2539  110   2649
03-spread         5195  180   5375
04-reckoning      1362   48   1410
05-lingering      1216   46   1262
All              11137  414  11551 

CLASS_DIRECTION  INEQUALITY_JUSTICE  INSTITUTION_ELITE    NA  OTHER  \
phase                                                                 
01-onset                         11                  0   825      0   
02-confirmation                  60                  5  2539      0   
03-spread                        94                  3  5195      3   
04-reckoning                     25                  1  1362      0   
05-lingering                     25                  0  1216      0   

CLASS_DIRECTION  VICTIMS_POOR  
phase                          
01-onset                   19  
02-confirmation            45  
03-spread                  80  
04-reckoning               22  
05-lingering               2

## Save

In [16]:
OUT.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT / "analysis_df.csv", index=False)
print(OUT.resolve() / "analysis_df.csv", df.shape)

C:\Users\PHSP8131\crisis-management-analysis\04-summary\output\ateneo-drowning\analysis_df.csv (11551, 93)
